# Ara25 — mBERT

Public research-preview notebook. Outputs and volatile metadata were removed. Set local dataset/output paths in the configuration cells before running.


In [ ]:
!pip install -U transformers datasets rouge-score nltk

In [ ]:
!pip install -q transformers datasets evaluate rouge-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q evaluate

In [ ]:
!pip install -q rouge-score

In [ ]:
!pip install -q sacrebleu

In [ ]:
# === mBERT Abstractive (BERT2BERT) — Two-Phase Training + Unicode-safe Decode + Safe Hierarchical Inference ===
import os, re, json, gc, math, random, numpy as np, pandas as pd, torch
from datetime import datetime
from datasets import Dataset
from inspect import signature
import evaluate
from transformers import (
    EncoderDecoderModel,
    BertTokenizerFast, BertConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    GenerationConfig, set_seed
)

# ---------------- Basics ----------------
def _slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "-", s).strip("-._")

MODEL_FAMILY = "mbert-abstractive-hier"
ENCODER_NAME = "bert-base-multilingual-cased"
DECODER_NAME = "bert-base-multilingual-cased"
SEED = 42

JSONL = "/content/drive/MyDrive/PACKED2x2_6144/train_ready_6144.jsonl"
SRC_CANDIDATES = ["Merged_Articles","merged_articles","model_input","text"]
TARGET_KEYS    = ["abstracted","target_summary","summary","target"]

# Inference budgets (not for training)
MAX_IN   = 6144
MAX_OUT  = 512

# Hierarchical inference
CHUNK_TOKS   = 480
CHUNK_STRIDE = 80
MID_OUT      = 160

APPLY_ARABIC_CONSTRAINTS = False
HIER_ON_VAL_AFTER_TRAIN  = False
N_VAL_HIER_SAMPLES       = 24

# Output dirs
_model_slug = _slug(ENCODER_NAME.split("/")[-1])
_ds_slug    = _slug(os.path.splitext(os.path.basename(JSONL))[0])
RUN_ID   = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{MODEL_FAMILY}-{_model_slug}-{_ds_slug}-{RUN_ID}"
OUT_ROOT = "/content/drive/MyDrive/sum_runs"
OUT      = f"{OUT_ROOT}/{RUN_NAME}"
for sub in ["checkpoints","logs","preds","metrics","artifacts"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("OUT:", OUT)

# ---------------- Env ----------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)
if torch.cuda.is_available():
    try: torch.cuda.empty_cache()
    except: pass

# ---------------- Data ----------------
with open(JSONL, "r", encoding="utf-8") as f:
    recs = [json.loads(l) for l in f if l.strip()]
assert recs, f"❌ لا توجد سجلات في {JSONL}"
df = pd.DataFrame(recs)

src_key = next((c for c in SRC_CANDIDATES if c in df.columns), None)
tgt_key = next((k for k in TARGET_KEYS    if k in df.columns), None)
assert src_key is not None, f"❌ لم أجد عمود المصدر من: {SRC_CANDIDATES}"
assert tgt_key is not None, f"❌ لم أجد عمود الملخص الهدف من: {TARGET_KEYS}"

_dup_open = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')
_tag_re   = re.compile(r'</?ARTICLE_\d+>')
def clean_src(s: str) -> str:
    s = _dup_open.sub(r'\1', s or "")
    s = _tag_re.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _tidy(s: str) -> str:
    if not s: return ""
    return re.sub(r"\s+", " ", s).strip()

ds_raw = Dataset.from_pandas(df, preserve_index=False)

# ---------------- Tokenizer & Model ----------------
tok = BertTokenizerFast.from_pretrained(ENCODER_NAME)
if tok.bos_token_id is None: tok.bos_token = tok.cls_token
if tok.eos_token_id is None: tok.eos_token = tok.sep_token

dec_cfg = BertConfig.from_pretrained(DECODER_NAME)
dec_cfg.is_decoder = True
dec_cfg.add_cross_attention = True

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    ENCODER_NAME, DECODER_NAME, decoder_config=dec_cfg
)

# Core config
model.config.pad_token_id = tok.pad_token_id
model.config.decoder_start_token_id = tok.bos_token_id
model.config.bos_token_id = tok.bos_token_id
model.config.eos_token_id = tok.eos_token_id
model.config.no_repeat_ngram_size = 3
model.config.num_beams = 4
model.config.early_stopping = True
model.config.use_cache = False
model.config.tie_word_embeddings = True
model.tie_weights()

forced_eos = tok.eos_token_id if tok.eos_token_id is not None else tok.sep_token_id
if hasattr(model, "generation_config"):
    gcfg = model.generation_config
    gcfg.pad_token_id = tok.pad_token_id
    gcfg.decoder_start_token_id = tok.bos_token_id
    gcfg.bos_token_id = tok.bos_token_id
    gcfg.eos_token_id = tok.eos_token_id
    gcfg.forced_eos_token_id = forced_eos
    gcfg.no_repeat_ngram_size = 3
    gcfg.num_beams = 4
    gcfg.length_penalty = 1.0

# Optional GC
for part in ["encoder","decoder"]:
    try: getattr(model, part).gradient_checkpointing_enable()
    except Exception: pass

model = model.to(device)
MODEL_MAX_CTX = int(getattr(model.config, "max_position_embeddings", 512))
EFF_MAX_IN_PHASE1 = min(448, MODEL_MAX_CTX)
EFF_MAX_IN_PHASE2 = min(512, MODEL_MAX_CTX)

# ---------------- Arabic constraints (optional) ----------------
bad_words_ids = None
if APPLY_ARABIC_CONSTRAINTS:
    ALLOWED_EXTRA = set(list(".,،؛:؟!()[]{}«»\"'…-–—%+*/=  \t\n\r"))
    def _is_ar_char(ch: str) -> bool:
        cp = ord(ch)
        return (0x0600 <= cp <= 0x06FF or 0x0750 <= cp <= 0x077F or
                0x08A0 <= cp <= 0x08FF or 0xFB50 <= cp <= 0xFDFF or
                0xFE70 <= cp <= 0xFEFF or ch in ALLOWED_EXTRA or ch == " ")
    _bad = []
    for tok_str, tok_id in tok.get_vocab().items():
        if tok_str in ["[PAD]"]:  # keep specials
            continue
        if any(not _is_ar_char(c) for c in tok_str):
            _bad.append(tok_id)
    bad_words_ids = [[i] for i in sorted(set(_bad)) if i not in {tok.pad_token_id, getattr(tok,"eos_token_id",-1)}]

# ---------------- Unicode-safe decode helpers ----------------
def _safe_argmax(arr):
    arr = np.asarray(arr)
    if arr.ndim >= 3:
        arr = arr.argmax(axis=-1)
    return arr

def _clip_to_vocab(arr, pad_id, vocab_size):
    arr = np.asarray(arr, dtype=np.int64)
    arr = np.where((arr < 0) | (arr >= vocab_size), pad_id, arr)
    return arr

def _safe_batch_decode(ids_2d, tok):
    out = []
    for row in ids_2d:
        row = [int(x) for x in row]
        try:
            out.append(tok.decode(row, skip_special_tokens=True))
        except Exception:
            row2 = [x for x in row if 0 <= x < tok.vocab_size]
            try:
                out.append(tok.decode(row2, skip_special_tokens=True))
            except Exception:
                out.append("")
    return out

# ---------------- Build DS ----------------
def build_ds(max_in_tokens: int):
    def tok_batch_local(batch):
        sources = [clean_src(s) for s in batch[src_key]]
        targets = [_tidy(t)       for t in batch[tgt_key]]
        mi = tok(sources, max_length=int(max_in_tokens), truncation=True, return_attention_mask=True)
        la = tok(text_target=targets, max_length=int(MAX_OUT), truncation=True)
        la_ids = [[(tid if tid != tok.pad_token_id else -100) for tid in seq] for seq in la["input_ids"]]
        mi["labels"] = la_ids
        return mi
    spl = ds_raw.train_test_split(test_size=0.10, seed=SEED)
    tr_raw, va_raw = spl["train"], spl["test"]
    tr = tr_raw.map(tok_batch_local, batched=True, remove_columns=tr_raw.column_names)
    va = va_raw.map(tok_batch_local, batched=True, remove_columns=va_raw.column_names)
    return tr, va, tr_raw, va_raw

# ---------------- Metrics (Unicode-safe) ----------------
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = _safe_argmax(preds)
    preds = _clip_to_vocab(preds, tok.pad_token_id, tok.vocab_size)
    labels = np.where(labels == -100, tok.pad_token_id, labels)
    labels = _clip_to_vocab(labels, tok.pad_token_id, tok.vocab_size)
    decoded_preds  = [_tidy(s) for s in _safe_batch_decode(preds, tok)]
    decoded_labels = [_tidy(s) for s in _safe_batch_decode(labels, tok)]
    res = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: float(res[k]) for k in res.keys()}

# ---------------- TrainingArguments helper (match save/eval) ----------------
def make_args(num_epochs: int, out_dir: str, gen_max_len: int, label_smoothing=0.1,
              per_bs=2, grad_acc=16, bf16_ok=False, strategy="epoch"):
    sig_args = signature(Seq2SeqTrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in sig_args else "evaluation_strategy"
    kwargs = dict(
        output_dir=out_dir,
        num_train_epochs=int(num_epochs),
        learning_rate=2e-5,
        warmup_ratio=0.2,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=0.5,
        per_device_train_batch_size=per_bs,
        per_device_eval_batch_size=per_bs,
        gradient_accumulation_steps=grad_acc,
        predict_with_generate=True,
        generation_num_beams=4,
        save_strategy=strategy,
        load_best_model_at_end=True,
        metric_for_best_model="rougeLsum",
        greater_is_better=True,
        logging_steps=20,
        logging_first_step=True,
        report_to=[],
        fp16=(not bf16_ok),
        bf16=bf16_ok,
        group_by_length=True,
        dataloader_num_workers=0,
        save_safetensors=True,
        label_smoothing_factor=label_smoothing,
    )
    if "generation_max_length" in sig_args:
        kwargs["generation_max_length"] = int(gen_max_len)
    if "dataloader_pin_memory" in sig_args:
        kwargs["dataloader_pin_memory"] = True
    kwargs[eval_key] = strategy
    if strategy == "steps":
        kwargs.update(dict(save_steps=1000, eval_steps=1000))
    return Seq2SeqTrainingArguments(**kwargs)

# ---------------- Optimizer param groups ----------------
from torch.optim import AdamW
def make_param_groups(model, lr_enc=1e-5, lr_dec=3e-5, wd=0.01):
    no_decay = ["bias", "LayerNorm.weight"]
    groups = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_dec = n.startswith("decoder")
        lr = lr_dec if is_dec else lr_enc
        wd_here = 0.0 if any(nd in n for nd in no_decay) else wd
        groups.append({"params":[p], "lr":lr, "weight_decay":wd_here})
    return groups

# ---------------- Phase A (freeze encoder) ----------------
for p in model.encoder.parameters():
    p.requires_grad = False

try:
    sm_ver = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
    use_bf16 = torch.cuda.is_available() and sm_ver >= 8
except Exception:
    use_bf16 = False

trA, vaA, tr_rawA, va_rawA = build_ds(EFF_MAX_IN_PHASE1)
coll = DataCollatorForSeq2Seq(tokenizer=tok, model=model, pad_to_multiple_of=8)
args_A = make_args(num_epochs=2, out_dir=OUT, gen_max_len=MAX_OUT, label_smoothing=0.1, bf16_ok=use_bf16, strategy="epoch")
optimizer_A = AdamW(make_param_groups(model, lr_enc=0.0, lr_dec=3e-5, wd=0.01), betas=(0.9,0.999), eps=1e-8)

trainer_A = Seq2SeqTrainer(
    model=model, args=args_A,
    train_dataset=trA, eval_dataset=vaA,
    data_collator=coll, compute_metrics=compute_metrics, tokenizer=tok,
    optimizers=(optimizer_A, None)
)
print(f"🚀 Phase A: decoder-only | EFF_MAX_IN={EFF_MAX_IN_PHASE1}")
trainer_A.train()

# ---------------- Phase B (unfreeze encoder) ----------------
for p in model.encoder.parameters():
    p.requires_grad = True

trB, vaB, tr_rawB, va_rawB = build_ds(EFF_MAX_IN_PHASE2)
args_B = make_args(num_epochs=6, out_dir=OUT, gen_max_len=MAX_OUT, label_smoothing=0.1, bf16_ok=use_bf16, strategy="epoch")
optimizer_B = AdamW(make_param_groups(model, lr_enc=1e-5, lr_dec=3e-5, wd=0.01), betas=(0.9,0.999), eps=1e-8)

trainer_B = Seq2SeqTrainer(
    model=model, args=args_B,
    train_dataset=trB, eval_dataset=vaB,
    data_collator=coll, compute_metrics=compute_metrics, tokenizer=tok,
    optimizers=(optimizer_B, None)
)
print(f"🚀 Phase B: unfreeze encoder | EFF_MAX_IN={EFF_MAX_IN_PHASE2}")
trainer_B.train()

# ---------------- Save best + GenConfig ----------------
best_dir = f"{OUT}/checkpoints/final_best_{RUN_NAME}"
trainer_B.save_model(best_dir)
tok.save_pretrained(best_dir)
gen_cfg = GenerationConfig(
    num_beams=4, do_sample=False, max_new_tokens=int(MAX_OUT), early_stopping=True,
    decoder_start_token_id=model.config.decoder_start_token_id,
    pad_token_id=model.config.pad_token_id,
    bos_token_id=model.config.bos_token_id,
    eos_token_id=model.config.eos_token_id,
    forced_eos_token_id=forced_eos,
    no_repeat_ngram_size=3, length_penalty=1.0,
)
gen_cfg.save_pretrained(best_dir)
print("✅ Saved BEST ->", best_dir)

# ---------------- Quick non-hier preds (Unicode-safe) ----------------
pred_out = trainer_B.predict(vaB)
preds_arr = pred_out.predictions[0] if isinstance(pred_out.predictions, tuple) else pred_out.predictions
preds_ids = _safe_argmax(preds_arr)
preds_ids = _clip_to_vocab(preds_ids, tok.pad_token_id, tok.vocab_size)
decoded_preds = [_tidy(s) for s in _safe_batch_decode(preds_ids, tok)]

labels_arr = np.where(pred_out.label_ids == -100, tok.pad_token_id, pred_out.label_ids)
labels_arr = _clip_to_vocab(labels_arr, tok.pad_token_id, tok.vocab_size)
decoded_refs = [_tidy(s) for s in _safe_batch_decode(labels_arr, tok)]

preds_df = pd.DataFrame({"id": list(range(len(decoded_preds))), "prediction": decoded_preds, "reference": decoded_refs})
preds_path = f"{OUT}/preds/preds_nonhier_{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
preds_df.to_csv(preds_path, index=False, encoding="utf-8-sig")
with open(f"{OUT}/metrics/val_metrics_nonhier_{RUN_NAME}.json","w",encoding="utf-8") as f:
    json.dump(pred_out.metrics, f, ensure_ascii=False, indent=2)
print("✅ Saved non-hier predictions ->", preds_path)

# ---------------- Inference (hierarchical up to 6144) ----------------
def _clip_ids_to_budget(ids, budget_tokens):
    return ids if len(ids) <= budget_tokens else ids[:budget_tokens]

def _chunk_input_by_tokens(text, chunk_size=CHUNK_TOKS, stride=CHUNK_STRIDE, user_budget=MAX_IN):
    ids_full = tok.encode(text, add_special_tokens=True)
    ids_full = _clip_ids_to_budget(ids_full, int(user_budget))
    chunks = []
    start = 0
    while start < len(ids_full):
        end = min(start + int(chunk_size), len(ids_full))
        sub = ids_full[start:end]
        if len(sub) > 0 and sub[-1] != tok.sep_token_id:
            sub = sub + [tok.sep_token_id]
        chunks.append(tok.decode(sub, skip_special_tokens=True))
        if end == len(ids_full): break
        start = max(0, end - int(stride))
        if len(chunks) >= 64: break
    return chunks

@torch.inference_mode()
def generate_simple(texts, max_new_tokens=128):
    inputs = tok(texts, max_length=EFF_MAX_IN_PHASE2, truncation=True, return_tensors="pt", padding=True).to(device)
    gen_kwargs = dict(num_beams=4, max_new_tokens=int(max_new_tokens), early_stopping=True, no_repeat_ngram_size=3)
    if bad_words_ids: gen_kwargs["bad_words_ids"] = bad_words_ids
    out = model.generate(**inputs, **gen_kwargs)
    return [_tidy(s) for s in tok.batch_decode(out, skip_special_tokens=True)]

@torch.inference_mode()
def hierarchical_summarize(text: str, mid_out=MID_OUT, final_out=MAX_OUT, user_budget=MAX_IN):
    base = clean_src(text)
    chunks = _chunk_input_by_tokens(base, CHUNK_TOKS, CHUNK_STRIDE, user_budget=user_budget)
    if not chunks:
        return generate_simple([base], max_new_tokens=int(final_out))[0], []
    inter_sums = [generate_simple([ch], max_new_tokens=int(mid_out))[0] for ch in chunks]
    bridge = " [SEP] ".join(inter_sums)
    final  = generate_simple([bridge], max_new_tokens=int(final_out))[0]
    return final, inter_sums

if HIER_ON_VAL_AFTER_TRAIN:
    sample_idx = list(range(min(N_VAL_HIER_SAMPLES, len(tr_rawB))))
    rows = []
    for i in sample_idx:
        src_txt = str(tr_rawB[i][src_key])
        ref_txt = _tidy(str(tr_rawB[i][tgt_key]))
        pred_txt, inter_list = hierarchical_summarize(src_txt, MID_OUT, MAX_OUT, user_budget=MAX_IN)
        rows.append({"id": int(i), "prediction": pred_txt, "reference": ref_txt, "n_inter_chunks": len(inter_list)})
    hier_df = pd.DataFrame(rows)
    hier_path = f"{OUT}/preds/preds_hier_sample_{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    hier_df.to_csv(hier_path, index=False, encoding="utf-8-sig")
    print("✅ Saved hierarchical sample preds ->", hier_path)

# ---------------- Manifest + cleanup ----------------
manifest = {
    "run_name": RUN_NAME,
    "run_id": RUN_ID,
    "out_dir": OUT,
    "encoder": ENCODER_NAME,
    "decoder": DECODER_NAME,
    "dataset_file": JSONL,
    "seed": SEED,
    "max_in_user": int(MAX_IN),
    "train_max_in_phase1": int(EFF_MAX_IN_PHASE1),
    "train_max_in_phase2": int(EFF_MAX_IN_PHASE2),
    "max_out": int(MAX_OUT),
    "chunk_tokens": int(CHUNK_TOKS), "chunk_stride": int(CHUNK_STRIDE),
    "mid_out": int(MID_OUT),
    "apply_arabic_constraints": bool(APPLY_ARABIC_CONSTRAINTS),
}
with open(f"{OUT}/artifacts/run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

del trA, vaA, trB, vaB, ds_raw, df, recs, preds_df
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("✅ ALL DONE")

In [ ]:
# === mBERT Hierarchical Generation (Clean Arabic) — GEN-ONLY | Robust BEST_DIR ===
import os, re, json, math, glob, torch, pandas as pd, numpy as np
from datetime import datetime
from typing import List, Dict, Any, Tuple
from tqdm.auto import tqdm
from transformers import BertTokenizerFast, EncoderDecoderModel

# ---------- إعداد المسارات ----------
BASE_DIR = "/content/drive/MyDrive/sum_runs"
BEST_DIR_OVERRIDE = ""  # إن أردت تحديد المسار يدويًا ضع المسار هنا كنص

def resolve_best_dir(base_dir: str, best_override: str = "") -> str:
    if best_override and os.path.isdir(best_override):
        return best_override
    # استخدم RUN_NAME إن كان موجودًا
    try:
        _rn = RUN_NAME  # من الخلايا السابقة
    except NameError:
        _rn = None
    if _rn:
        cand = f"{base_dir}/{_rn}/checkpoints/final_best_{_rn}"
        if os.path.isdir(cand):
            return cand
    # ابحث عن أحدث final_best_* داخل جميع تشغيلات mBERT
    runs = [d for d in glob.glob(os.path.join(base_dir, "mbert-abstractive-hier-*")) if os.path.isdir(d)]
    bests = []
    for r in runs:
        bests.extend([p for p in glob.glob(os.path.join(r, "checkpoints", "final_best_*")) if os.path.isdir(p)])
    if not bests:
        raise FileNotFoundError(f"❌ لم أجد أي مجلد final_best_* داخل {base_dir}")
    bests.sort(key=os.path.getmtime, reverse=True)
    return bests[0]

BEST_DIR = resolve_best_dir(BASE_DIR, BEST_DIR_OVERRIDE)
RUN_DIR  = os.path.dirname(os.path.dirname(BEST_DIR))   # .../run/checkpoints/final_best_* -> .../run
RUN_NAME = os.path.basename(RUN_DIR)
PRED_DIR = f"{RUN_DIR}/preds"; os.makedirs(PRED_DIR, exist_ok=True)

print("✅ Using BEST_DIR:", BEST_DIR)

# مجموعة الفحص الخاصة بك:
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"

# ---------- حدود التلخيص ----------
MAX_IN        = 6144
MAX_OUT       = 512
CHUNK_TOKS    = 480
CHUNK_STRIDE  = 80
MID_OUT       = 160
MAX_CHUNKS    = 64

NUM_BEAMS          = 4
NO_REPEAT_NGRAM    = 3
LENGTH_PENALTY     = 1.05
REPETITION_PENALTY = 1.15
MIN_NEW_TOKENS     = 40

AR_ONLY = True  # منع التوكنات غير العربية (اختياري)

device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True

# ---------- تحميل البيانات ----------
def load_jsonl(path: str):
    recs=[]
    with open(path,"r",encoding="utf-8-sig",errors="replace") as f:
        for ln in f:
            s=ln.strip()
            if not s or s.startswith("```"): continue
            try: recs.append(json.loads(s))
            except: pass
    return recs

def get_source(rec: Dict[str,Any]) -> str:
    return (rec.get("Merged_Articles") or rec.get("merged_articles") or
            rec.get("model_input") or rec.get("text") or "")

items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

# ---------- تنظيف عربي ----------
TAG_RE  = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)
SPC_RE  = re.compile(r"\s+")
DIAC_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
TAT_RE  = re.compile(r"\u0640+")

def normalize_arabic(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    s = SPC_RE.sub(" ", s).strip()
    return s

def clean_source(s: str) -> str:
    s = normalize_arabic(s)
    s = TAT_RE.sub("", s)
    s = DIAC_RE.sub("", s)
    return s

def clean_output(s: str) -> str:
    s = normalize_arabic(s)
    s = TAT_RE.sub("", s)
    s = DIAC_RE.sub("", s)
    parts = re.split(r"(?<=[\.!\؟\!]|[،؛])\s+", s)
    seen, out = set(), []
    for p in parts:
        q = p.strip()
        if q and q not in seen:
            seen.add(q); out.append(q)
    s = " ".join(out)
    s = re.sub(r"(\b\w+\b)(\s+\1){2,}", r"\1", s)
    s = re.sub(r"[،,:;]+$", "", s).strip()
    return s

# ---------- تحميل الموديل/التوكنيزر محليًا ----------
tok = BertTokenizerFast.from_pretrained(BEST_DIR, local_files_only=True)
if tok.bos_token_id is None and tok.cls_token_id is not None: tok.bos_token = tok.cls_token
if tok.eos_token_id is None and tok.sep_token_id is not None: tok.eos_token = tok.sep_token

model = EncoderDecoderModel.from_pretrained(BEST_DIR, local_files_only=True)
cfg = model.config
cfg.pad_token_id              = tok.pad_token_id
cfg.decoder_start_token_id    = tok.bos_token_id or tok.cls_token_id or tok.pad_token_id
cfg.bos_token_id              = tok.bos_token_id or tok.cls_token_id or tok.pad_token_id
cfg.eos_token_id              = tok.eos_token_id or tok.sep_token_id or tok.pad_token_id
cfg.num_beams                 = NUM_BEAMS
cfg.no_repeat_ngram_size      = NO_REPEAT_NGRAM
cfg.early_stopping            = True
cfg.use_cache                 = False

if hasattr(model, "generation_config"):
    gcg = model.generation_config
    gcg.pad_token_id = cfg.pad_token_id
    gcg.decoder_start_token_id = cfg.decoder_start_token_id
    gcg.bos_token_id = cfg.bos_token_id
    gcg.eos_token_id = cfg.eos_token_id
    gcg.num_beams = cfg.num_beams
    gcg.no_repeat_ngram_size = cfg.no_repeat_ngram_size
    gcg.early_stopping = True

model = model.to(device).eval()

# ---------- قيود عربية (WordPiece-aware) ----------
def build_bad_words_ids(tokenizer: BertTokenizerFast):
    if not AR_ONLY: return None
    ALLOWED_EXTRA = set(list(".,،؛:؟!()[]{}«»\"'…-–—%+*/=  \t\n\r"))
    ranges = [(0x0600,0x06FF),(0x0750,0x077F),(0x08A0,0x08FF),(0xFB50,0xFDFF),(0xFE70,0xFEFF)]
    def _is_ar(ch):
        cp = ord(ch)
        return any(lo<=cp<=hi for lo,hi in ranges) or ch in ALLOWED_EXTRA
    def _strip_wp(s): return s[2:] if s.startswith("##") else s
    v = tokenizer.get_vocab()
    specials = {tokenizer.pad_token, tokenizer.sep_token, tokenizer.cls_token, tokenizer.mask_token, tokenizer.unk_token}
    special_ids = {i for s,i in v.items() if s in specials}
    bad = []
    for ts, tid in v.items():
        if tid in special_ids: continue
        s = _strip_wp(ts)
        if not s: continue
        if any(not _is_ar(c) for c in s):
            bad.append(tid)
    return [[i] for i in sorted(set(bad)) if i not in {tokenizer.pad_token_id, cfg.eos_token_id}]

bad_words_ids = build_bad_words_ids(tok)

# ---------- تقطيع هرمي ----------
MODEL_MAX_CTX = int(getattr(cfg, "max_position_embeddings", 512))
EFF_MAX_IN    = min(MODEL_MAX_CTX, 512)

def _clip_budget(ids: List[int], budget: int) -> List[int]:
    return ids if len(ids) <= budget else ids[:budget]

def _chunk_input_ids(text: str, chunk_size=CHUNK_TOKS, stride=CHUNK_STRIDE, max_chunks=MAX_CHUNKS, user_budget=MAX_IN) -> List[str]:
    ids = tok.encode(text or "", add_special_tokens=True)
    ids = _clip_budget(ids, int(user_budget))
    chunks = []
    start = 0; n = 0
    while start < len(ids) and n < int(max_chunks):
        end = min(start + int(chunk_size), len(ids))
        sub = ids[start:end]
        if len(sub) > 0 and sub[-1] != tok.sep_token_id and tok.sep_token_id is not None:
            sub = sub + [tok.sep_token_id]
        chunks.append(tok.decode(sub, skip_special_tokens=True))
        if end == len(ids): break
        start = max(0, end - int(stride))
        n += 1
    return chunks

# ---------- توليد ----------
@torch.inference_mode()
def _generate_simple(texts: List[str], max_new_tokens=128) -> List[str]:
    mnt = int(min(max_new_tokens, MAX_OUT))
    mnt = max(MIN_NEW_TOKENS, mnt)
    enc = tok(texts, max_length=EFF_MAX_IN, truncation=True, return_tensors="pt", padding=True).to(device)
    out = model.generate(
        **enc,
        num_beams=NUM_BEAMS,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        early_stopping=True,
        max_new_tokens=mnt,
        min_new_tokens=MIN_NEW_TOKENS,
        length_penalty=float(LENGTH_PENALTY),
        repetition_penalty=float(REPETITION_PENALTY),
        bad_words_ids=bad_words_ids,
        decoder_start_token_id=cfg.decoder_start_token_id,
        pad_token_id=cfg.pad_token_id,
        bos_token_id=cfg.bos_token_id,
        eos_token_id=cfg.eos_token_id,
    )
    return tok.batch_decode(out, skip_special_tokens=True)

@torch.inference_mode()
def hierarchical_summarize(text: str, mid_out=MID_OUT, final_out=MAX_OUT, user_budget=MAX_IN) -> Tuple[str, list]:
    base = clean_source(text)
    chunks = _chunk_input_ids(base, CHUNK_TOKS, CHUNK_STRIDE, MAX_CHUNKS, user_budget=user_budget)
    if not chunks:
        return clean_output(_generate_simple([base], max_new_tokens=int(final_out))[0]), []
    inter = [_generate_simple([c], max_new_tokens=int(mid_out))[0] for c in chunks]
    inter = [clean_output(s) for s in inter]
    bridge = " [SEP] ".join(inter)
    final  = clean_output(_generate_simple([bridge], max_new_tokens=int(final_out))[0])
    return final, inter

# ---------- تشغيل على مجموعة الفحص وحفظ CSV ----------
rows = []
for i, rec in tqdm(list(enumerate(items)), total=len(items), desc="Generating (mBERT Hier + Clean Arabic)"):
    src = get_source(rec)
    pred, inter_list = hierarchical_summarize(src, MID_OUT, MAX_OUT, user_budget=MAX_IN)
    rows.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "prediction": pred,
        "n_inter_chunks": len(inter_list),
    })

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pred_path = f"{PRED_DIR}/preds_hier_clean_{RUN_NAME}_{ts}.csv"
pd.DataFrame(rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

print("✅ Saved predictions ->", pred_path)
print(f"📦 Count: {len(rows)} | Run: {RUN_NAME} | Beams={NUM_BEAMS} | mid={MID_OUT} | final={MAX_OUT} | AR_ONLY={AR_ONLY}")


In [ ]:
# =================== Comprehensive Evaluation for mBERT Hier Generation ===================
!pip -q install "pandas>=2.0" "evaluate==0.4.2" "rouge_score==0.1.2" "sacrebleu==2.4.0" \
                 "bert-score==0.3.13" "transformers>=4.41.0" "pot>=0.9.3" tqdm

import os, re, json, math, glob
from typing import List, Dict, Any, Tuple
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
import evaluate
from bert_score import BERTScorer
import ot  # POT

# --------- Locate run & predictions ---------
BASE_DIR = "/content/drive/MyDrive/sum_runs"
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH_OVERRIDE = ""  # اختياري: ضع مسار CSV محدد إن أردت

def resolve_run_dir(base_dir: str) -> str:
    try:
        rn = RUN_NAME
    except NameError:
        rn = None
    if rn:
        rd = f"{base_dir}/{rn}"
        if os.path.isdir(rd): return rd
    # fallback: latest mbert run
    runs = [d for d in glob.glob(os.path.join(base_dir, "mbert-abstractive-hier-*")) if os.path.isdir(d)]
    assert runs, f"❌ لا توجد تشغيلات داخل {base_dir}"
    runs.sort(key=os.path.getmtime, reverse=True)
    return runs[0]

RUN_DIR = resolve_run_dir(BASE_DIR)
PRED_DIR = f"{RUN_DIR}/preds"
OUT_DIR  = f"{RUN_DIR}/metrics/eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

def latest_pred_csv(pred_dir: str) -> str:
    cands = []
    cands.extend(glob.glob(os.path.join(pred_dir, "preds_hier_clean_*.csv")))
    if not cands:
        cands.extend(glob.glob(os.path.join(pred_dir, "*.csv")))
    assert cands, f"❌ لم أجد ملفات CSV داخل {pred_dir}"
    cands.sort(key=os.path.getmtime, reverse=True)
    return cands[0]

PRED_PATH = PRED_PATH_OVERRIDE if PRED_PATH_OVERRIDE else latest_pred_csv(PRED_DIR)

print("✅ RUN_DIR :", RUN_DIR)
print("📝 PRED   :", PRED_PATH)
print("🧪 TEST   :", TEST_PATH)
print("📊 OUT    :", OUT_DIR)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("🚀 Device:", device)

# --------- Helpers ---------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

# --------- Load test + refs ---------
items = load_jsonl(TEST_PATH)
assert items, f"❌ لا يوجد عناصر في {TEST_PATH}"
print("🧪 Loaded items:", len(items))

ref1 = [sanitize_text(x.get("target_summary", "")) for x in items]
ref2 = [sanitize_text(x.get("target_summary_2", "")) for x in items]
has_r2 = [bool(r.strip()) for r in ref2]
num_r2 = sum(1 for x in has_r2 if x)
print(f"Ref1 count: {len(ref1)} | Ref2 non-empty: {num_r2}/{len(ref2)}")

# --------- Load predictions CSV ---------
dfp = pd.read_csv(PRED_PATH)
pred_col = None
for c in ["prediction","pred","output","generated","summary","generated_summary"]:
    if c in dfp.columns: pred_col = c; break
assert pred_col is not None, f"❌ لم أجد عمود التنبؤات في CSV. الأعمدة: {list(dfp.columns)}"

# mapping by id if exists
id2pred = {}
if "id" in dfp.columns:
    for _, row in dfp.iterrows():
        id2pred[str(row["id"])] = sanitize_text(str(row[pred_col]))
    preds = []
    for i, rec in enumerate(items):
        key = str(rec.get("id", i))
        preds.append(id2pred.get(key, ""))
else:
    preds = [sanitize_text(x) for x in dfp[pred_col].astype(str).tolist()]
    # pad if shorter
    if len(preds) < len(items):
        preds += [""] * (len(items) - len(preds))

print("✅ Predictions loaded:", len(preds))
nonempty = sum(1 for p in preds if p.strip())
avg_words = float(np.mean([len(p.split()) for p in preds])) if preds else 0.0
print(f"🔎 Pred diagnostics -> non-empty: {nonempty}/{len(preds)} | avg_words: {avg_words:.1f}")

# --------- Build refs structures ---------
refs_multi = []
for i in range(len(items)):
    rlist = []
    if ref1[i].strip(): rlist.append(ref1[i])
    if ref2[i].strip(): rlist.append(ref2[i])
    if not rlist: rlist = [""]  # ضمان وجود مرجع
    refs_multi.append(rlist)

# =================== ROUGE ===================
rouge = evaluate.load("rouge")
def rouge_scores(pred, ref):
    sc = rouge.compute(predictions=pred, references=ref, use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

rouge_ref1 = rouge_scores(preds, ref1)
rouge_ref2 = rouge_scores(preds, [r if r else "" for r in ref2]) if num_r2>0 else {k: None for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

# best-over-refs (ROUGE-Lsum per-sample)
rouge1_list = []; rouge2_list = []; rougeL_list = []; rougeLsum_list = []; best_ref_ix=[]
for i in tqdm(range(len(items)), desc="ROUGE best-over-refs"):
    rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
    best = {"r1":0.0,"r2":0.0,"rL":0.0,"rLsum":0.0}; bix=0
    for j, r in enumerate(rs):
        sc = rouge.compute(predictions=[preds[i]], references=[r], use_stemmer=False)
        if sc["rougeLsum"] > best["rLsum"]:
            best = {"r1":float(sc["rouge1"]), "r2":float(sc["rouge2"]), "rL":float(sc["rougeL"]), "rLsum":float(sc["rougeLsum"])}
            bix=j
    rouge1_list.append(best["r1"]); rouge2_list.append(best["r2"])
    rougeL_list.append(best["rL"]); rougeLsum_list.append(best["rLsum"])
    best_ref_ix.append(bix)
rouge_best = {
    "rouge1": float(np.mean(rouge1_list)),
    "rouge2": float(np.mean(rouge2_list)),
    "rougeL": float(np.mean(rougeL_list)),
    "rougeLsum": float(np.mean(rougeLsum_list)),
}
rouge_avg = {k: None if rouge_ref2[k] is None else (rouge_ref1[k] + rouge_ref2[k]) / 2.0 for k in rouge_ref1}

# =================== SacreBLEU / chrF ===================
sacrebleu = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def wrap_single(refs):  # [[ref_i] ...]
    return [[r] for r in refs]

bleu_multi = sacrebleu.compute(predictions=preds, references=refs_multi)["score"]
bleu_r1    = sacrebleu.compute(predictions=preds, references=wrap_single(ref1))["score"]
bleu_r2    = sacrebleu.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
bleu_avg   = None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0

chrf_multi = chrf_metric.compute(predictions=preds, references=refs_multi)["score"]
chrf_r1    = chrf_metric.compute(predictions=preds, references=wrap_single(ref1))["score"]
chrf_r2    = chrf_metric.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
chrf_avg   = None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0

# =================== METEOR ===================
try:
    meteor_metric = evaluate.load("meteor")
    met_r1 = meteor_metric.compute(predictions=preds, references=ref1)["meteor"]
    met_r2 = meteor_metric.compute(predictions=preds, references=[r if r else "" for r in ref2])["meteor"] if num_r2>0 else None
    # best-over-refs
    met_best_list = []
    for i in tqdm(range(len(items)), desc="METEOR best-over-refs"):
        rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
        best = 0.0
        for r in rs:
            best = max(best, meteor_metric.compute(predictions=[preds[i]], references=[r])["meteor"])
        met_best_list.append(best)
    met_best = float(np.mean(met_best_list))
except Exception as e:
    print("METEOR skipped:", e)
    met_r1 = met_r2 = met_best = None

# =================== BERTScore (per-ref + best) ===================
print("\n🎯 BERTScore (per-ref + best)")
BERT_MODEL = "xlm-roberta-large"
scorer = BERTScorer(model_type=BERT_MODEL, lang=None, rescale_with_baseline=False, idf=True, device=device, batch_size=8)

# احسب IDF من جميع المراجع المتاحة
all_refs_flat = [r for rs in refs_multi for r in rs]
scorer.compute_idf(all_refs_flat)

def bert_over(preds, refs):
    P,R,F = scorer.score(preds, refs)
    return float(torch.mean(P).item()), float(torch.mean(R).item()), float(torch.mean(F).item())

bP1,bR1,bF1 = bert_over(preds, ref1)
if num_r2>0:
    bP2,bR2,bF2 = bert_over(preds, [r if r else "" for r in ref2])
else:
    bP2=bR2=bF2=None

P1,R1,F1 = scorer.score(preds, ref1)
if num_r2>0:
    P2,R2,F2 = scorer.score(preds, [r if r else "" for r in ref2])
bP_best=[]; bR_best=[]; bF_best=[]
for i in range(len(items)):
    if num_r2>0 and F2[i] > F1[i]:
        bP_best.append(float(P2[i])); bR_best.append(float(R2[i])); bF_best.append(float(F2[i]))
    else:
        bP_best.append(float(P1[i])); bR_best.append(float(R1[i])); bF_best.append(float(F1[i]))
bP_best = float(np.mean(bP_best)); bR_best = float(np.mean(bR_best)); bF_best = float(np.mean(bF_best))

# =================== MoverScore (OT) ===================
print("\n⚖️  MoverScore (per-ref + best)")
EMB_MODEL = "xlm-roberta-large"
MAX_LEN   = 512
tok_emb = AutoTokenizer.from_pretrained(EMB_MODEL)
mdl_emb = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()

@torch.no_grad()
def embed_tokens(text: str, max_len: int = MAX_LEN) -> Tuple[torch.Tensor, List[int]]:
    out = tok_emb(text or "", return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=True)
    out = {k: v.to(device) for k, v in out.items()}
    hs = mdl_emb(**out).last_hidden_state.squeeze(0)
    ids = out["input_ids"].squeeze(0).tolist()
    special = set(t for t in [getattr(tok_emb, "cls_token_id", None),
                              getattr(tok_emb, "sep_token_id", None),
                              getattr(tok_emb, "pad_token_id", None)] if t is not None)
    keep = [i for i, tid in enumerate(ids) if tid not in special] or list(range(len(ids)))
    return hs[keep], [ids[i] for i in keep]

def build_idf_from_texts(texts: List[str], max_len: int = MAX_LEN) -> dict:
    from collections import Counter, defaultdict
    df_counter = Counter()
    for t in texts:
        ids = tok_emb(t or "", truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
        df_counter.update(set(ids))
    N = max(1, len(texts))
    idf = defaultdict(lambda: 0.0)
    for tid, dfv in df_counter.items():
        idf[tid] = math.log((N + 1) / (dfv + 1))
    return idf

def cosine_cost(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    Xn = torch.nn.functional.normalize(X, dim=1)
    Yn = torch.nn.functional.normalize(Y, dim=1)
    sim = Xn @ Yn.T
    return (1.0 - sim).clamp(min=0.0)

idf_pred = build_idf_from_texts(preds)
idf_ref1 = build_idf_from_texts(ref1)
idf_ref2 = build_idf_from_texts([r if r else "" for r in ref2]) if num_r2>0 else {}

def moverscore_pair(hyp: str, ref: str, idf_h: dict, idf_r: dict) -> float:
    H, ids_h = embed_tokens(hyp, MAX_LEN)
    R, ids_r = embed_tokens(ref, MAX_LEN)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_h.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_r.get(tid, 0.0) for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy()
    b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    T = ot.emd(a, b, C)
    cost = float((T * C).sum())
    return max(0.0, min(1.0, 1.0 - cost))

ms_r1, ms_r2, ms_best = [], [], []
for i in tqdm(range(len(items)), desc="MoverScore"):
    p = preds[i]
    r1 = ref1[i] if ref1[i] else ""
    s1 = moverscore_pair(p, r1, idf_pred, idf_ref1)
    ms_r1.append(s1)
    if num_r2>0 and ref2[i]:
        s2 = moverscore_pair(p, ref2[i], idf_pred, idf_ref2)
        ms_r2.append(s2)
        ms_best.append(max(s1, s2))
    else:
        ms_best.append(s1)

mover_ref1 = float(np.mean(ms_r1))
mover_ref2 = float(np.mean(ms_r2)) if num_r2>0 and ms_r2 else None
mover_best = float(np.mean(ms_best))

# =================== Summary & Save ===================
summary = {
    "run_dir": RUN_DIR,
    "preds_path": PRED_PATH,
    "refs_path": TEST_PATH,
    "count": len(items),
    "pred_non_empty": nonempty,
    "pred_avg_words": avg_words,
    # ROUGE
    "rouge1_ref1_mean": rouge_ref1["rouge1"],
    "rouge2_ref1_mean": rouge_ref1["rouge2"],
    "rougeL_ref1_mean": rouge_ref1["rougeL"],
    "rougeLsum_ref1_mean": rouge_ref1["rougeLsum"],
    "rouge1_ref2_mean": rouge_ref2["rouge1"] if rouge_ref2["rouge1"] is not None else None,
    "rouge2_ref2_mean": rouge_ref2["rouge2"] if rouge_ref2["rouge2"] is not None else None,
    "rougeL_ref2_mean": rouge_ref2["rougeL"] if rouge_ref2["rougeL"] is not None else None,
    "rougeLsum_ref2_mean": rouge_ref2["rougeLsum"] if rouge_ref2["rougeLsum"] is not None else None,
    "rouge1_best_over_refs": rouge_best["rouge1"],
    "rouge2_best_over_refs": rouge_best["rouge2"],
    "rougeL_best_over_refs": rouge_best["rougeL"],
    "rougeLsum_best_over_refs": rouge_best["rougeLsum"],
    # BLEU / chrF
    "sacrebleu_multi_ref": bleu_multi,
    "sacrebleu_ref1": bleu_r1,
    "sacrebleu_ref2": bleu_r2,
    "sacrebleu_avg": None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0,
    "chrf_multi_ref": chrf_multi,
    "chrf_ref1": chrf_r1,
    "chrf_ref2": chrf_r2,
    "chrf_avg": None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0,
    # METEOR
    "meteor_ref1": met_r1,
    "meteor_ref2": met_r2,
    "meteor_best_over_refs": None if 'met_best' not in locals() else met_best,
    # BERTScore
    "bertscore_ref1": {"P": bP1, "R": bR1, "F1": bF1},
    "bertscore_ref2": ({"P": bP2, "R": bR2, "F1": bF2} if bF2 is not None else None),
    "bertscore_best_over_refs": {"P": bP_best, "R": bR_best, "F1": bF_best},
    # MoverScore
    "moverscore_ref1": mover_ref1,
    "moverscore_ref2": mover_ref2,
    "moverscore_best_over_refs": mover_best,
}

rows = []
for i in range(len(items)):
    row = {
        "id": items[i].get("id", i),
        "title": items[i].get("title", ""),
        "prediction": preds[i],
        "ref1": ref1[i],
        "ref2": ref2[i],
        "rougeLsum_ref1": rouge.compute(predictions=[preds[i]], references=[ref1[i] or ""], use_stemmer=False)["rougeLsum"] if ref1[i] else 0.0,
        "rougeLsum_ref2": rouge.compute(predictions=[preds[i]], references=[ref2[i] or ""], use_stemmer=False)["rougeLsum"] if ref2[i] else 0.0,
        "rougeLsum_best": rougeLsum_list[i],
        "best_ref": "Ref1" if (best_ref_ix[i]==0 or not ref2[i]) else "Ref2",
        "moverscore_best": ms_best[i],
    }
    rows.append(row)

csv_path = os.path.join(OUT_DIR, "eval_report_per_sample.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "eval_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== Overall Metrics ===")
for k,v in summary.items():
    if k in ["preds_path","refs_path","run_dir","count"]:
        pass
    print(f"{k}: {v}")

print(f"\n📄 Saved per-sample CSV -> {csv_path}")
print(f"📊 Saved summary JSON   -> {os.path.join(OUT_DIR, 'eval_summary.json')}")
print("✅ Done.")


In [ ]:
# === Top-5 Summaries (Print to Colab) — best by ROUGE-Lsum over Ref1/Ref2 ===
!pip -q install "rouge_score>=0.1.2" "pandas>=2.0"

import os, re, json, glob, pandas as pd
from datetime import datetime
from rouge_score import rouge_scorer

# -------- مساراتك --------
BASE_DIR = "/content/drive/MyDrive/sum_runs"
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH_OVERRIDE = ""  # ضع مسار CSV يدويًا إن رغبت

# -------- مساعدات --------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)
def sanitize(s: str) -> str:
    s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str):
    recs=[]
    with open(path,"r",encoding="utf-8-sig",errors="replace") as f:
        for ln in f:
            s=ln.strip()
            if not s or s.startswith("```"): continue
            try: recs.append(json.loads(s))
            except: pass
    return recs

def resolve_run_dir(base_dir: str):
    try:
        rn = RUN_NAME
    except NameError:
        rn = None
    if rn and os.path.isdir(f"{base_dir}/{rn}"):
        return f"{base_dir}/{rn}"
    runs = [d for d in glob.glob(os.path.join(base_dir,"mbert-abstractive-hier-*")) if os.path.isdir(d)]
    assert runs, f"❌ لا توجد تشغيلات داخل {base_dir}"
    runs.sort(key=os.path.getmtime, reverse=True)
    return runs[0]

def latest_pred_csv(pred_dir: str):
    cands = glob.glob(os.path.join(pred_dir, "preds_hier_clean_*.csv")) or glob.glob(os.path.join(pred_dir, "*.csv"))
    assert cands, f"❌ لم أجد ملفات CSV داخل {pred_dir}"
    cands.sort(key=os.path.getmtime, reverse=True)
    return cands[0]

# -------- تحضير البيانات --------
RUN_DIR = resolve_run_dir(BASE_DIR)
PRED_DIR = f"{RUN_DIR}/preds"
PRED_PATH = PRED_PATH_OVERRIDE or latest_pred_csv(PRED_DIR)

items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

dfp = pd.read_csv(PRED_PATH)
pred_col = next((c for c in ["prediction","pred","output","summary","generated","generated_summary"] if c in dfp.columns), None)
assert pred_col, f"❌ لم أجد عمود التنبؤات في CSV. الأعمدة: {list(dfp.columns)}"

# اصطفاف التنبؤات مع عناصر الاختبار
if "id" in dfp.columns:
    id2pred = {str(r["id"]): sanitize(str(r[pred_col])) for _,r in dfp.iterrows()}
    preds = [id2pred.get(str(rec.get("id", i)), "") for i,rec in enumerate(items)]
else:
    preds = [sanitize(x) for x in dfp[pred_col].astype(str).tolist()]
    if len(preds) < len(items): preds += [""]*(len(items)-len(preds))
    preds = preds[:len(items)]

ref1 = [sanitize(x.get("target_summary","")) for x in items]
ref2 = [sanitize(x.get("target_summary_2","")) for x in items]
titles = [sanitize(x.get("title","")) for x in items]

# -------- احسب ROUGE-Lsum واختَر الأفضل --------
scorer = rouge_scorer.RougeScorer(["rougeLsum"], use_stemmer=False)
rows = []
for i,(p,r1,r2) in enumerate(zip(preds, ref1, ref2)):
    s1 = scorer.score(r1 or "", p or "")["rougeLsum"].fmeasure if r1 else 0.0
    s2 = scorer.score(r2 or "", p or "")["rougeLsum"].fmeasure if r2 else 0.0
    best = max(s1, s2); best_ref = "Ref1" if s1 >= s2 else "Ref2"
    rows.append({
        "idx": i, "title": titles[i], "pred": p, "ref1": r1, "ref2": r2,
        "rougeLsum_ref1": s1, "rougeLsum_ref2": s2, "rougeLsum_best": best, "best_ref": best_ref
    })

rows.sort(key=lambda r: r["rougeLsum_best"], reverse=True)
top5 = rows[:5]

# -------- طباعة منسقة على شاشة كولاب --------
print(f"\n====== Top-5 by ROUGE-Lsum (best over refs) ======")
print(f"Run: {os.path.basename(RUN_DIR)} | File: {os.path.basename(PRED_PATH)}\n")
for k, r in enumerate(top5, 1):
    print(f"#{k} | idx={r['idx']} | best_ref={r['best_ref']} | rougeLsum_best={r['rougeLsum_best']:.4f}")
    if r["title"]:
        print("• العنوان:", r["title"])
    print(f"• ROUGE-Lsum Ref1: {r['rougeLsum_ref1']:.4f} | Ref2: {r['rougeLsum_ref2']:.4f}")
    print("— المرجع 1 —")
    print(r["ref1"] if r["ref1"] else "—")
    print("— المرجع 2 —")
    print(r["ref2"] if r["ref2"] else "—")
    print("— المُلخّص المُولَّد —")
    print(r["pred"] if r["pred"] else "—")
    print("-"*100)
